In [29]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import catboost as cb


train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")
sample_sub = pd.read_csv("/kaggle/input/playground-series-s6e2/sample_submission.csv")

TARGET = "Heart Disease"
ID_COL = "id"


cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [col for col in cat_cols if col != TARGET]


for col in cat_cols:
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)



X = train.drop([TARGET, ID_COL], axis=1)
y = train[TARGET]

X_test = test.drop([ID_COL], axis=1)



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_preds = np.zeros(len(X_test))
cb_preds = np.zeros(len(X_test))
oof = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n========== Fold {fold+1} ==========")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # =========================
    # LightGBM
    # =========================
    
    lgb_model = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.01,
        num_leaves=128,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=50,
        random_state=42
    )
    
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100)]
    )
    
    oof[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    lgb_preds += lgb_model.predict_proba(X_test)[:, 1] / skf.n_splits
    
 
    
    cb_model = cb.CatBoostClassifier(
        iterations=3000,
        learning_rate=0.01,
        depth=6,
        l2_leaf_reg=3,
        eval_metric="AUC",
        random_seed=42,
        verbose=200
    )
    
    cb_model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        cat_features=cat_cols,
        early_stopping_rounds=100
    )
    
    cb_preds += cb_model.predict_proba(X_test)[:, 1] / skf.n_splits




print("\nFinal OOF AUC:", roc_auc_score(y, oof))



final_preds = 0.5 * lgb_preds + 0.5 * cb_preds



sample_sub[TARGET] = final_preds
sample_sub.to_csv("submission.csv", index=False)

print("\nSubmission file created successfully!")



========== Fold 1 ==========
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.067805 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 422
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 13
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1373]	valid_0's auc: 0.955313	valid_0's binary_logloss: 0.268149
0:	test: 0.9404809	best: 0.9404809 (0)	total: 81.6ms	remaining: 4m 4s
200:	test: 0.9527628	best: 0.9527628 (200)	total: 14.4s	remaining: 3m 21s
400:	te